# Часть 2.5 — Dairy Products 2025 (China)

Данные по молочной отрасли китайских провинций. Колонки на китайском — переведу в латиницу для удобства.

3 эксперимента: (1) регрессия `gdp_contribution` от метрик отрасли; (2) регрессия цены `price` от спроса/предложения; (3) KMeans-кластеризация провинций по профилю.

In [1]:
from pathlib import Path

import kagglehub
import pandas as pd
from _setup import evaluate_models, make_spark
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator, RegressionEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F

SEED = 42
spark = make_spark("hw3-dairy")
spark

## Загрузка

In [2]:
csv = next(Path(kagglehub.dataset_download("tangke111/dairy-product-related-datasets-for-2025")).glob("*.csv"))
rename = {
    "省份": "province",
    "牛奶产量（吨）": "milk",
    "酸奶产量（吨）": "yogurt",
    "奶酪产量（吨）": "cheese",
    "黄油产量（吨）": "butter",
    "乳制品消费量（吨）": "consumption",
    "乳制品进口量（吨）": "imports",
    "乳制品出口量（吨）": "exports",
    "乳制品库存（吨）": "stock",
    "乳制品价格（元/公斤）": "price",
    "牧场数量（个）": "farms",
    "奶牛数量（万头）": "cows",
    "人均乳制品消费量（公斤/年）": "per_capita",
    "乳制品加工厂数量（个）": "plants",
    "城市化率（%）": "urbanization",
    "乳业从业人员（万人）": "workers",
    "奶源基地面积（万亩）": "farm_area",
    "乳制品行业GDP贡献（亿元）": "gdp_contribution",
}
raw = spark.read.csv(str(csv), header=True, inferSchema=True)
df = raw.toDF(*[rename.get(c, c) for c in raw.columns]).cache()
print(f"строк: {df.count()}, колонок: {len(df.columns)}")
df.show(3)

строк: 10500, колонок: 18
+------------+------+------+------+------+-----------+-------+-------+------+-----+-----+----+----------+------+------------+-------+---------+----------------+
|    province|  milk|yogurt|cheese|butter|consumption|imports|exports| stock|price|farms|cows|per_capita|plants|urbanization|workers|farm_area|gdp_contribution|
+------------+------+------+------+------+-----------+-------+-------+------+-----+-----+----+----------+------+------------+-------+---------+----------------+
|内蒙古自治区| 99908| 15768|  7727| 19581|     970542| 274759|  13782|114677| 5.21|  381|11.9|      35.0|    48|        74.2|    4.6|    734.2|          492.14|
|      山东省|721410| 50948| 15839| 11903|    1355760| 151382|   8104|343430|16.05|  365|28.5|      30.4|   267|        31.7|   15.9|    948.1|           10.24|
|      陕西省|516856|148563| 10649| 13232|     426285| 120061|  23422|329315|17.56|  351| 2.2|      29.5|   146|        68.2|   14.1|    720.7|          133.58|
+------------+------

## Подготовка

In [3]:
feature_cols = [c for c in df.columns if c not in ("province", "gdp_contribution", "price")]
prep = Pipeline(
    stages=[
        VectorAssembler(inputCols=feature_cols, outputCol="features_raw"),
        StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True),
    ]
).fit(df)
data = (
    prep.transform(df)
    .withColumn("label_gdp", F.col("gdp_contribution").cast("double"))
    .withColumn("label_price", F.col("price").cast("double"))
    .select("features", "label_gdp", "label_price", "province", *feature_cols)
)
train, test = data.randomSplit([0.8, 0.2], seed=SEED)
train.cache()
test.cache()
print(f"train: {train.count()}, test: {test.count()}")

train: 8477, test: 2023


## Эксперимент 1 — регрессия gdp_contribution

In [4]:
ev_gdp = {
    "RMSE": RegressionEvaluator(labelCol="label_gdp", metricName="rmse"),
    "R2": RegressionEvaluator(labelCol="label_gdp", metricName="r2"),
}
models_gdp = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="label_gdp", maxIter=50),
    "RFRegressor": RandomForestRegressor(
        featuresCol="features", labelCol="label_gdp", numTrees=100, seed=SEED, maxDepth=8
    ),
}
evaluate_models(models_gdp, train, test, ev_gdp)

,RMSE,R2
model,,
LinearRegression,145.589683,-0.004921
RFRegressor,145.833692,-0.008293


## Эксперимент 2 — регрессия price

In [5]:
ev_price = {
    "RMSE": RegressionEvaluator(labelCol="label_price", metricName="rmse"),
    "R2": RegressionEvaluator(labelCol="label_price", metricName="r2"),
}
models_price = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="label_price", maxIter=50),
    "RFRegressor": RandomForestRegressor(
        featuresCol="features", labelCol="label_price", numTrees=100, seed=SEED, maxDepth=8
    ),
}
evaluate_models(models_price, train, test, ev_price)

,RMSE,R2
model,,
LinearRegression,4.298648,-0.001983
RFRegressor,4.298147,-0.001749


## Эксперимент 3 — KMeans (k=2..6)

In [6]:
sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
rows = []
for k in range(2, 7):
    m = KMeans(k=k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
    rows.append({"k": k, "silhouette": sil_eval.evaluate(m.transform(data)), "WSSSE": m.summary.trainingCost})
exp3 = pd.DataFrame(rows).set_index("k")
exp3

,silhouette,WSSSE
k,,
2,0.083997,150449.415476
3,0.075367,145517.238339
4,0.076650,141317.509509
5,0.074559,138340.491585
6,0.074172,135624.710782


In [7]:
best_k = int(exp3["silhouette"].idxmax())
print(f"best k = {best_k}")
best = KMeans(k=best_k, featuresCol="features", seed=SEED, maxIter=20).fit(data)
best.transform(prep.transform(df)).groupBy("prediction").agg(
    F.count("*").alias("n"),
    F.avg("milk").alias("avg_milk"),
    F.avg("consumption").alias("avg_consumption"),
    F.avg("price").alias("avg_price"),
    F.avg("gdp_contribution").alias("avg_gdp"),
).orderBy("prediction").show()

best k = 2


+----------+----+------------------+------------------+------------------+------------------+
|prediction|   n|          avg_milk|   avg_consumption|         avg_price|           avg_gdp|
+----------+----+------------------+------------------+------------------+------------------+
|         0|5322|1007806.1223224351|1277149.9727546035|12.426708004509585|250.58353250657595|
|         1|5178|1033477.1616454229|1308072.3211664734| 12.46768250289688|256.17322904596375|
+----------+----+------------------+------------------+------------------+------------------+



## Выводы

- **Эксп. 1 и 2**: обе регрессии (gdp_contribution, price) дают **R² ≈ 0** — фичи практически не объясняют целевую. Похоже, датасет искусственный: значения по всем колонкам имеют одинаковое распределение независимо от провинции.
- **Эксп. 3**: silhouette 0.07–0.08, чёткой структуры нет; best k=2 даёт два почти идентичных кластера (avg_milk, avg_price, avg_gdp совпадают до второго знака).
- Главный вывод: сигнала в данных нет — supervised и unsupervised методы бессильны. На реальном датасете тех же колонок ожидался бы сильный R² (молочное производство объясняет GDP молочной отрасли почти линейно).

In [8]:
spark.stop()